# Build a Receiver

This notebook presents the receiver as a set of stages you can inspect: antenna input, mixer output, IF filter, and audio output. Instead of a clickable diagram, it uses a stage selector that exposes the waveform and spectrum at each point.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Stage-by-Stage Inspection

When you choose a stage below, you are looking at the same signal after different receiver operations have been applied.

In [ ]:
fs = 192_000
t = np.arange(0, 0.04, 1 / fs)
message = 0.8 * np.sin(2 * np.pi * 1500 * t) + 0.3 * np.sin(2 * np.pi * 500 * t)
rf = fm_modulate(message, carrier_freq=52_000, fs=fs, freq_dev=2500)
adjacent = 0.5 * fm_modulate(0.7 * np.sin(2 * np.pi * 2300 * t), carrier_freq=63_000, fs=fs, freq_dev=1800)
front_end = rf + adjacent
lo = np.cos(2 * np.pi * 40_000 * t)
mixed = front_end * lo
if_sos = signal.butter(5, [10_000, 14_000], btype="band", fs=fs, output="sos")
if_signal = signal.sosfilt(if_sos, mixed)
audio = fm_demodulate(if_signal, fs=fs, audio_cutoff=4000)

stages = {
    "RF front end": front_end,
    "Mixer output": mixed,
    "IF filter": if_signal,
    "Recovered audio": audio,
}


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_stage(stage_name="RF front end"):
    sig = stages[stage_name]
    axes[0].clear()
    axes[1].clear()
    plot_waveform(sig[:6000], fs=fs, ax=axes[0], title=stage_name)
    plot_spectrum(sig, fs=fs, ax=axes[1], title=f"{stage_name} spectrum")
    axes[1].set_xlim(0, 80_000 if stage_name != "Recovered audio" else 10_000)
    axes[1].set_ylim(-110, 5)
    fig.canvas.draw_idle()
    if stage_name == "Recovered audio":
        refresh_audio_widget(audio_out, resample_signal(sig, fs, 44_100), rate=44_100)
    else:
        with audio_out:
            audio_out.clear_output(wait=True)
            display(Markdown("Select **Recovered audio** to hear the demodulated result."))

controls = widgets.interactive(
    update_stage,
    stage_name=dropdown(options=list(stages.keys()), value="RF front end", description="Stage"),
)
display(controls, audio_out)


## Key Takeaway

Receiver design is easier to understand when you stop thinking of a radio as a black box. Each stage changes what information is obvious and what information is suppressed.